# Day 1 -- From pixels to vision transformers

Welcome. Over the next three hours you will build **five image classifiers** on the same
medical images and watch each one beat the last:

> **logistic regression -> MLP -> CNN -> ResNet -> Vision Transformer**

Each rung up the ladder adds one idea, and you will *see* the accuracy climb because of it.
That climb is the whole story of how computer vision got good.

### The dataset and the task
We use **APTOS-2019**: color photographs of the back of the eye (the *retina*, or *fundus*).
Diabetes damages the tiny blood vessels there, a disease called **diabetic retinopathy (DR)**.
Our task is the one actually deployed in clinics: **referable DR** -- does this eye show
enough disease that the person should see a specialist? **Yes or no.** A binary screening call.

### What you'll be able to do by the end
- Explain what an image *is* to a model, and why pixels alone are hard.
- Say what each model adds: non-linearity, spatial structure, pretraining, attention.
- Read a **learning curve** and a **confusion matrix**, and know why accuracy can mislead.
- Understand *transfer learning* -- the single most useful trick in applied vision.

### How the lab works
Fill in the `# TODO` lines. Stuck? Ask Claude, then make sure you understand the answer
before moving on. The point isn't to finish fastest, it's to understand *why each rung
beats the one below it.*

In [ ]:
# Setup: on Colab, grab the course files. Locally (already in the repo) this is a no-op.
import os, sys
if not os.path.exists("common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day1_ladder")
sys.path.insert(0, ".")
sys.path.insert(0, "../_shared")
import colab_setup
colab_setup.ensure()
colab_setup.gpu_check()

In [ ]:
import os, sys
import torch
import numpy as np
import common
# nbfig lives in notebooks/_shared; add it relative to common.py so this cell
# works even if the setup cell above wasn't run, and from any working directory.
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(common.__file__)), "..", "_shared"))
import nbfig            # Colab-safe branded plotting (matches the slide figures)
nbfig.use()

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
results = {}   # we'll collect each model's validation accuracy here

## 0. The data: pixels, eyes, and a yes/no question

Before any modeling, spend real time looking at the data. Good practitioners always do.

### 0.1 The clinical task
About **1 in 3** people with diabetes develop some retinopathy, and it is a leading cause
of blindness in working-age adults, yet it is **preventable** if caught early. The catch:
catching it needs a trained eye specialist, and there are far too few. That mismatch -- huge
routine need, scarce expertise -- is exactly where AI screening helps. Let's look at the eyes.

In [ ]:
train_loader, val_loader = common.get_loaders(size=224, batch_size=32)
images, labels = next(iter(train_loader))
print("one batch:", images.shape, "  <- (batch, channels, height, width)")

# Look at eight real fundus images with their referable / not-referable labels.
fig, axes = nbfig.fig(2, 4, figsize=(11, 5.6))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(common._denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(common.GRADE_NAMES[int(lab)], fontsize=11,
                 color=(nbfig.DEEPPINK if int(lab) else nbfig.INK))
    ax.axis("off")
nbfig.show(fig, "Eight eyes: can you tell which need a doctor?")

### 0.2 An image is just numbers
To a model, this photo is not a picture -- it's a grid of numbers. A *color* image is
**three** grids stacked: how much red, green, and blue at each pixel. Everything today
operates on those numbers.

In [ ]:
common.show_rgb_split(images[0])      # the same eye as red / green / blue grids
common.show_pixel_histogram(images[0])  # the distribution of pixel brightnesses

### 0.3 Normalization: why the model sees "weird" colors
Networks train best when their inputs are centered around zero. So before an image goes in,
we **normalize** each color channel: subtract a mean and divide by a standard deviation (the
classic ImageNet values). The picture below shows the eye you'd recognize, the *exact tensor
the model actually receives* (rescaled so we can view it -- note the color shift), and the
per-channel means before vs after. After normalization the means sit near zero. That's the
whole point.

In [ ]:
raw = common._denorm(images[0])     # back to the 0..1 image a human recognizes
normed = images[0]                  # the zero-centered tensor the model is fed
disp = (normed - normed.min()) / (normed.max() - normed.min())  # rescale just to view

fig, (a1, a2, a3) = nbfig.fig(1, 3, figsize=(11, 3.6))
a1.imshow(raw.permute(1, 2, 0).numpy()); a1.set_title("raw (0..1)", fontsize=11); a1.axis("off")
a2.imshow(disp.permute(1, 2, 0).numpy()); a2.set_title("normalized (what the model sees)", fontsize=11); a2.axis("off")
a3.grid(True)
x = np.arange(3)
a3.bar(x - 0.2, raw.mean((1, 2)).numpy(), 0.4, color=nbfig.TURQUOISE, label="raw")
a3.bar(x + 0.2, normed.mean((1, 2)).numpy(), 0.4, color=nbfig.DEEPPINK, label="normalized")
a3.axhline(0, color=nbfig.MUTED, lw=0.8); a3.set_xticks(x); a3.set_xticklabels(["R", "G", "B"])
a3.set_title("channel means", fontsize=11); a3.legend()
nbfig.show(fig, "Normalization centers every channel near zero")

### 0.4 Know your labels (this matters more than you think)
Before trusting any accuracy number, check how many examples of each class you have. If
most eyes are "not referable," a lazy model can score high by always guessing the majority.
Hold that thought -- we'll catch a model doing exactly that at the end.

In [ ]:
import numpy as np
# count classes across the validation set
all_labels = np.concatenate([y.numpy() for _, y in val_loader])
counts = [int((all_labels == c).sum()) for c in range(common.NUM_CLASSES)]

fig, ax = nbfig.fig(figsize=(5.5, 3.2))
bars = ax.bar(common.GRADE_NAMES, counts, color=[nbfig.TURQUOISE, nbfig.DEEPPINK])
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width() / 2, c, str(c), ha="center", va="bottom",
            fontweight="bold", family="DejaVu Sans Mono")
ax.set_ylabel("validation images")
nbfig.show(fig, "Class balance")
print("majority-class baseline accuracy:", f"{max(counts) / sum(counts):.3f}")

### 0.5 Augmentation: turn one eye into many
We have only a few thousand images. **Augmentation** makes small random changes -- flip,
rotate -- every time an image is used, so the model sees more variety and can't just
memorize specific photos. Here is the *same eye*, fifteen times, run through the real
training pipeline. Each one is a slightly different training example, for free.

In [ ]:
import torchvision.transforms as T
pil = T.ToPILImage()(common._denorm(images[0]))
aug = common._transform(224, train=True)   # the actual training-time augmentation

fig, axes = nbfig.fig(3, 5, figsize=(11, 7))
for ax in axes.ravel():
    shown = common._denorm(aug(pil))        # apply augmentation, then de-normalize to view
    ax.imshow(shown.permute(1, 2, 0).numpy()); ax.axis("off")
nbfig.show(fig, "The same eye, 15 random augmentations")

## 1. Logistic regression -- the simplest classifier

We start at the bottom rung. Flatten each image into one long row of numbers and fit a
**linear** model: it draws a single straight boundary between "referable" and "not." We
use small 64x64 images so it runs in seconds.

**Predict first:** a coin flip gets ~50%, and always-guessing-the-majority gets the
number you just printed. How much better can a straight line on raw pixels do? Write a guess.

In [ ]:
from sklearn.linear_model import LogisticRegression

tr64, va64 = common.get_loaders(size=64, batch_size=64)
Xtr, ytr = common.flatten_for_classical(tr64)
Xva, yva = common.flatten_for_classical(va64)
print("flattened features per image:", Xtr.shape[1])

# TODO: make a LogisticRegression with max_iter=500 and solver='saga'
# TODO: fit the classifier on the training features Xtr, ytr

preds = clf.predict(Xva)
acc = (preds == yva).mean()
results["logreg"] = acc
print(f"logistic regression val accuracy: {acc:.3f}")

### 1.1 Look at *how* it's right and wrong
Accuracy is one number; a **confusion matrix** shows the mistakes. Rows are the truth,
columns are the guess. The diagonal is correct; off-diagonal is where it slips.

In [ ]:
nbfig.confusion(yva, preds, common.GRADE_NAMES, text="Logistic regression").show()

Logistic regression treats every pixel as independent and can only draw straight-line
boundaries. It has no idea that neighboring pixels form vessels, spots, or shapes. Hold
that thought -- it's the ceiling every later model breaks through.

## 2. Multi-layer perceptron -- stacking non-linear layers

Same flattened pixels, but now we stack layers with non-linearities between them, so the
model can **bend** its decision boundary instead of using one straight cut. It still has
no notion of space, though -- shuffle the pixels and it wouldn't notice.

In [ ]:
# TODO: build an MLP with common.make_mlp; in_features is 3*64*64 (the flattened size)
# TODO: train it: common.train_model(mlp, tr64, va64, epochs=5, lr=1e-3, device=device)
results["mlp"] = history[-1][1]
print(f"mlp val accuracy: {history[-1][1]:.3f}")

### 2.1 The learning curve
Unlike logreg, a neural net learns *gradually*, one epoch (one pass through the data) at a
time. The curve below shows validation accuracy climbing while the training loss falls --
that's learning happening. A flat curve means it's stuck; a falling-then-rising-loss curve
would mean trouble.

In [ ]:
nbfig.learning_curve(history, text="MLP: accuracy up, loss down").show()

*Did it beat logreg by much?* Usually only a little. Extra layers help the model bend, but
it still can't see that pixels next to each other belong together. That missing idea --
**spatial structure** -- is the next rung, and it's a big one.

## 3. Convolutional neural network -- seeing in 2D

A **CNN** slides small filters across the image, so it understands *spatial* structure:
edges, blobs, textures, and where they are. Each filter is a little pattern-detector that
fires when it sees its pattern anywhere in the image. Now we switch to the full 224x224 images.

**Predict:** how much will accuracy jump versus the MLP?

In [ ]:
tr224, va224 = common.get_loaders(size=224, batch_size=32)

# TODO: build the CNN with common.make_small_cnn()
# TODO: train it for 15 epochs at lr=1e-3 on the 224px loaders
results["cnn"] = history[-1][1]
print(f"cnn val accuracy: {history[-1][1]:.3f}")

In [ ]:
nbfig.learning_curve(history, text="CNN: a steeper climb").show()

### 3.1 What did it learn to see?
We can peek at the filters in the very first convolutional layer. Nobody told the network
what to look for -- it *learned* these little edge and color detectors on its own, just
from trying to predict the label.

In [ ]:
common.show_first_layer_filters(cnn)

### 3.2 Its mistakes

In [ ]:
res = common.evaluate_classifier(lambda b: cnn(b).argmax(1), va224, device)
nbfig.confusion(res["y"], res["pred"], common.GRADE_NAMES, text="CNN confusion matrix").show()

That jump over the MLP is convolutions earning their keep: the model can finally use the
*arrangement* of pixels, not just their values. But we trained it from scratch on a few
thousand images. The next rung borrows a head start from millions.

## 4. ResNet50 -- transfer learning

Instead of learning vision from scratch, we take a 50-layer network already trained on a
million everyday photos (ImageNet) and **reuse its vision.** We freeze it and train only a
small new final layer for our yes/no question. This is the single biggest practical trick
in modern computer vision.

![Transfer learning: reuse the backbone, train a tiny head](img/transfer_learning.png)

In [ ]:
# TODO: build a pretrained ResNet50 with common.make_resnet50(pretrained=True)
# TODO: finetune for 3 epochs at lr=1e-3 on the 224px loaders
results["resnet"] = history[-1][1]
print(f"resnet val accuracy: {history[-1][1]:.3f}")

In [ ]:
nbfig.learning_curve(history, text="ResNet: high accuracy in just 3 epochs").show()

### 4.1 Where is it looking?
A real worry in medical AI: is the model looking at the *disease*, or at some artifact (a
bright edge, the camera label)? **Grad-CAM** highlights the pixels that most drove the
prediction. We want the heat on the retina, not the border.

In [ ]:
import numpy as np
from PIL import Image

imgs, _ = next(iter(va224))
cam, predicted = common.gradcam(resnet, imgs[0], device=device)
cam_up = np.array(Image.fromarray((cam * 255).astype("uint8")).resize((224, 224))) / 255.0

fig, (a1, a2) = nbfig.fig(1, 2, figsize=(8.5, 4.4))
base = common._denorm(imgs[0]).permute(1, 2, 0).numpy()
a1.imshow(base); a1.set_title("the eye", fontsize=11); a1.axis("off")
a2.imshow(base); a2.imshow(cam_up, cmap="inferno", alpha=0.5)
a2.set_title(f"where ResNet looked (pred: {common.GRADE_NAMES[predicted]})", fontsize=11)
a2.axis("off")
nbfig.show(fig, "Grad-CAM: the model's attention")

*Why did pretraining help so much with so little training?* The network already knew how to
see edges, textures, and shapes. We only had to teach it which combinations mean "referable."
That's the power of standing on a million photos' worth of prior learning.

## 5. Vision Transformer -- patches that pay attention

The newest rung. A **ViT** chops the image into patches, turns each patch into a vector, and
lets the patches *pay attention* to each other -- deciding which patches matter for the call.
It's the exact machinery behind the language models you've used, just patches instead of
words. (Much more on that tomorrow.)

![A ViT: image to patches to attention to prediction](img/vit_patches.png)

In [ ]:
# TODO: build a pretrained ViT with common.make_vit_base(pretrained=True)
# TODO: train the head for 5 epochs at lr=1e-3 on the 224px loaders
results["vit"] = history[-1][1]
print(f"vit val accuracy: {history[-1][1]:.3f}")

In [ ]:
nbfig.learning_curve(history, text="Vision Transformer").show()

On a few thousand images the ViT and the ResNet usually land close together -- transformers
are hungrier for data, so their real advantage shows at larger scale. The point isn't that
ViT always wins; it's that *attention* is a second, equally powerful way to see, and it's the
bridge to tomorrow.

## 6. The leaderboard you just built

Five models, same data, same yes/no question. Watch the climb -- and remember which idea
bought each step up.

In [ ]:
names = list(results.keys())
accs = [results[n] for n in names]

fig, ax = nbfig.fig(figsize=(7.5, 4))
bars = ax.bar(names, accs, color=nbfig.palette(len(names)))
for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width() / 2, a + 0.01, f"{a:.2f}", ha="center",
            fontweight="bold", family="DejaVu Sans Mono")
ax.set_ylabel("validation accuracy"); ax.set_ylim(0, 1)
nbfig.show(fig, "Same data, five models")

Each rung added exactly one idea: **non-linearity** (MLP), **spatial structure** (CNN),
**pretraining** (ResNet), **attention** (ViT). That is, in miniature, the last fifteen years
of computer vision.

## 7. Accuracy can lie -- and why that's dangerous here

Remember the class balance from section 0.3. In screening, the classes are lopsided and the
**costs are not symmetric**: telling a sick person they're fine (a *false negative*) is far
worse than a false alarm. A model can post a high accuracy while quietly missing the cases
that matter most. Always read the confusion matrix, not just the headline number.

In [ ]:
best = common.evaluate_classifier(lambda b: resnet(b).argmax(1), va224, device)
nbfig.confusion(best["y"], best["pred"], common.GRADE_NAMES,
                text="Best model: count the missed 'referable' eyes").show()
print("per-class recall:", {k: round(v, 2) for k, v in
      common.evaluate_classifier(lambda b: resnet(b).argmax(1), va224, device)["per_class"].items()})

The bottom-left cell -- truly *referable* eyes the model called *not referable* -- is the one
a clinician loses sleep over. This is why medical AI is judged on sensitivity and specificity,
not raw accuracy. We dig into exactly that vocabulary in the slides.

## Bridge to Day 2

The Vision Transformer worked by splitting the image into patches and letting them attend to
each other:

```
   IMAGE                                     SENTENCE
   [patch][patch][patch]  --embed-->         "the"  "cat"  "sat"  --embed-->
        vectors                                    vectors
          |                                          |
       attention  (which patches matter?)         attention  (which words matter?)
          |                                          |
       prediction: referable?                     prediction: next word
```

That second column is a **Large Language Model**. Same machinery, different input. Tomorrow we
use one to read radiology reports and combine it with images. See you then.

## Stretch -- find a disagreement

If you finished early:

1. Find a validation eye where the **ResNet and the ViT predict differently**. Show the image
   and both predictions. What's unusual about it -- blurry, dark, an artifact?
2. Run Grad-CAM on a model's **wrong** prediction. Was it looking at the wrong thing?
3. Which model would you actually deploy in a clinic, and why? (Hint: it's not just accuracy.)